In [ ]:
from huggingface_hub import snapshot_download

# This will download the entire repository to a folder named 'herald_local'
model_path = snapshot_download(
    repo_id="FrenzyMath/Herald_translator",
    local_dir="./herald_local",
    local_dir_use_symlinks=False  # Copies files directly instead of using symlinks
)

print(f"Model downloaded to: {model_path}")

In [1]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Clear previous memory usage (Very Important if you just had an OOM)
def clear_vram():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

clear_vram()

model_id = "./herald_local"  # Path to your downloaded model

# 2. Configure 4-bit (NF4) Quantization
# This configuration is the gold standard for 12GB cards.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # High-accuracy 4-bit
    bnb_4bit_use_double_quant=True,      # Second pass quantization to save ~0.4 bits/param
    bnb_4bit_compute_dtype=torch.float16 # Computation still happens in 16-bit for speed
)

# 3. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,           # Initial load weight reduction
    low_cpu_mem_usage=True,              # Prevents RAM spikes
    trust_remote_code=True
)

# 4. Optimized Translation Function
def translate_to_lean(nl_statement):
    # This specific template matches the Herald dataset's training style
    prompt = f"Instruction: Translate the following natural language mathematical statement into Lean 4.\nInput: {nl_statement}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=256,
            do_sample=False,   # Keep it deterministic for math
            temperature=0,     # Standard for formalization tasks
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Slice the output to remove the prompt part
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("Output:")[-1].strip()

/home/magn3144/Documents/Github/AlphaProof/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 273/273 [00:04<00:00, 68.22it/s, Materializing param=model.norm.weight]                               


In [5]:
# Example test
if __name__ == "__main__":
    test_math = r"A sequence of real numbers $u_n$ converges to a limit $L$ if for every $\epsilon > 0$, there exists an $N$ such that for all $n \ge N$, $|u_n - L| < \epsilon$."
    print(f"Lean 4 Output:\n{translate_to_lean(test_math)}")

Lean 4 Output:
import Mathlib
open Filter Function TopologicalSpace Topology Set UniformSpace Uniformity

/-- A sequence of real numbers $u_n$ converges to a limit $L$ if for every $\epsilon > 0$, there exists an $N$ such that for all $n \ge N$, $|u_n - L| < \epsilon$. -/
theorem tendsto_nhds_real {u : ℕ → ℝ} {L : ℝ} :
    Tendsto u atTop (𝓝 L) ↔ ∀ ε > 0, ∃ N : ℕ, ∀ n ≥ N, |u n - L| < ε := sorry


In [1]:
from nobodywho import Chat

# Load your newly created 4-bit GGUF model
chat = Chat("./herald_q4_k_m.gguf")

def translate_to_lean(nl_statement):
    prompt = f"Instruction: Translate to Lean 4.\nInput: {nl_statement}\nOutput:"
    
    # NobodyWho handles tokenization and GPU offloading (via Vulkan/Metal) automatically
    response = chat.ask(prompt).completed()
    return response.strip()

print(translate_to_lean("The sum of two even numbers is even."))

import Mathlib
open Multiplicative

/-- The sum of two even numbers is even. -/
theorem add_two_even {n m : ℤ} : Even n → Even m → Even (n + m) :=  by sorry
